Imports

In [2]:
import os
import re
import urllib3
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin, parse_qsl
import time
from io import BytesIO
from PIL import Image

In [3]:
scraped_data = "./data"
os.makedirs(scraped_data, exist_ok=True)

In [4]:
def fix_url_schema(url: str) -> str:
    """Fixes common URL typos such as missing slashes ('https:/domain.com' -> 'https://domain.com')."""
    if not url or not isinstance(url, str):
        return ""

    cleaned_url = url.strip().replace("\n", "").replace("\r", "")

    # Fix single slash after protocol scheme (e.g., http:/ or https:/)
    cleaned_url = re.sub(r"^(https?):/(?!/)", r"\1://", cleaned_url)

    # Prepend https:// if no scheme is provided
    if not cleaned_url.startswith(("http://", "https://")):
        cleaned_url = "https://" + cleaned_url

    return cleaned_url

Cleaning & Sanitizing Url

In [5]:
def sanitize_and_generate_dynamic_folder(raw_url: str) -> tuple:
    """Cleans trailing whitespaces, fixes broken URL slashes, and generates safe folder paths."""
    sanitized_url = fix_url_schema(raw_url)
    if not sanitized_url:
        return None, None

    parsed = urlparse(sanitized_url)
    netloc = parsed.netloc.lower()

    # Fallback if domain parsing fails
    if not netloc:
        netloc = "unknown_domain"

    # Generate folder name without risky characters
    query_dict = dict(parse_qsl(parsed.query, keep_blank_values=False))
    domain_clean = netloc.replace("www.", "")
    id_marker = query_dict.get("id") or query_dict.get("oga_service") or "index"

    # Safe Windows/Linux folder slug conversion
    id_marker_clean = (
        re.sub(r"[^a-zA-Z0-9_\-]", "_", str(id_marker))[:30].strip("_")
    )
    final_folder_name = f"{domain_clean}_{id_marker_clean}"

    # Target path inside base scraped_data folder
    unique_target_path = os.path.normpath(
        os.path.join(scraped_data, final_folder_name)
    )

    return sanitized_url, unique_target_path

Extract text, images and documents from each webpage


In [6]:
def dynamic_session_extractor(target_url: str, session_folder_path: str):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Cache-Control": "max-age=0",
        "Connection": "keep-alive"
    }
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    # Setup targeted sub-folder paths
    image_folder = os.path.join(session_folder_path, "images")
    doc_folder = os.path.join(session_folder_path, "documents")
    
    try:
        with requests.get(target_url, headers=headers, stream=True, timeout=20, verify=False) as response:
            if response.status_code == 404:
                print("   ⚠ 404 Server Status: Target rejected or record missing.")
                return
            response.raise_for_status()
            
            content_type = response.headers.get('Content-Type', '').lower()
            content_disposition = response.headers.get('Content-Disposition', '')
            
            is_binary = any(t in content_type for t in ['pdf', 'spreadsheet', 'excel', 'word', 'octet-stream', 'zip'])
            is_forced = 'filename=' in content_disposition
            
            # --- SCENARIO A: Direct Document Streams (PDF, Word, Excel) ---
            if is_binary or is_forced:
                if 'text/html' in content_type: return
                
                if is_forced:
                    filename = re.findall(r'filename=["\']?(.*?)["\']?$', content_disposition)[0]
                else:
                    filename = os.path.basename(urlparse(target_url).path)
                    
                if not filename or not filename.lower().endswith(('.pdf', '.xlsx', '.xls', '.docx', '.doc')):
                    ext = '.pdf' if 'pdf' in content_type else '.xlsx' if 'excel' in content_type else '.docx'
                    filename = f"document_download{ext}"
                
                # Force-route to the sub-folder
                os.makedirs(doc_folder, exist_ok=True)
                with open(os.path.join(doc_folder, filename), 'wb') as f:
                    for chunk in response.iter_content(chunk_size=16384): 
                        f.write(chunk)
                print(f"   ✓ Extracted direct document file: {filename} -> Saved to sub-folder")
                
            # --- SCENARIO B: HTML Page Parsing ---
            else:
                os.makedirs(session_folder_path, exist_ok=True)
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Decompose clutter & forms
                for element in soup(["script", "style", "nav", "footer", "header", "aside", "form", "iframe", "noscript"]):
                    element.decompose()
                
                # Prevent "Continue reading" navigational noise
                nav_noise_patterns = re.compile(r'^(continue reading|read more|click here|learn more|view more|go to.*|back to top)$', re.IGNORECASE)
                for noisy_text_element in soup.find_all(['a', 'button', 'span', 'p']):
                    text_content = noisy_text_element.get_text(strip=True)
                    if nav_noise_patterns.match(text_content):
                        noisy_text_element.decompose()

                structured_blocks = []
                tags_to_extract = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'ul', 'ol', 'table']
                all_elements = soup.find_all(tags_to_extract)
                
                for element in all_elements:
                    if element.find_parent(['table', 'ul', 'ol']): continue
                    
                    if element.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
                        text = element.get_text(strip=True)
                        if text: structured_blocks.append(f"\n{text}\n")
                    elif element.name == 'p':
                        text = element.get_text(strip=True)
                        if text: structured_blocks.append(text)
                    elif element.name in ['ul', 'ol']:
                        list_items = [f"- {li.get_text(strip=True)}" for li in element.find_all('li', recursive=False) if li.get_text(strip=True)]
                        if list_items: structured_blocks.append("\n" + "\n".join(list_items) + "\n")
                    elif element.name == 'table':
                        headers_list = []
                        columns_data = {}
                        thead = element.find('thead')
                        header_source = thead if thead else element
                        first_tr = header_source.find('tr')
                        if first_tr:
                            for th in first_tr.find_all(['th', 'td']):
                                header_text = th.get_text(strip=True)
                                headers_list.append(header_text)
                                columns_data[header_text] = []
                        tbody = element.find('tbody')
                        row_source = tbody if tbody else element
                        all_trs = row_source.find_all('tr')
                        start_idx = 1 if not thead and first_tr in all_trs else 0
                        for tr in all_trs[start_idx:]:
                            for idx, td in enumerate(tr.find_all(['td', 'th'])):
                                if idx < len(headers_list):
                                    cell_strings = [s for s in td.stripped_strings]
                                    cell_text = " ".join(cell_strings) if cell_strings else "-"
                                    columns_data[headers_list[idx]].append(cell_text)
                        table_output = [f"{header}: {', '.join(columns_data[header])}" for header in headers_list if header]
                        if table_output: structured_blocks.append("\n" + "\n".join(table_output) + "\n")

                # Save Text Payload
                final_page_text = "\n".join(structured_blocks).strip()
                text_filepath = os.path.join(session_folder_path, "content.txt")
                with open(text_filepath, "w", encoding="utf-8") as text_file:
                    text_file.write(f"Source URL: {target_url}\n" + "="*80 + "\n\n" + final_page_text)
                print(f"   ✓ Extracted page content text and tables -> Saved.")

                # Documents Link Discovery & Extract into dedicated sub-folder
                anchors = soup.find_all('a', href=True)
                doc_extensions = re.compile(r'\.(pdf|xlsx|xls|docx|doc)(\?|$)', re.IGNORECASE)
                valid_links = set()
                
                for link in anchors:
                    href = link['href']
                    if doc_extensions.search(href):
                        if not any(term in href.lower() for term in ['login', 'logout', 'javascript:', 'facebook', 'twitter']):
                            valid_links.add(urljoin(target_url, href))
                
                for asset_route in valid_links:
                    try:
                        with requests.get(asset_route, headers=headers, stream=True, timeout=15, verify=False) as sub_res:
                            sub_type = sub_res.headers.get('Content-Type', '').lower()
                            if 'text/html' in sub_type or not sub_res.ok: continue
                            
                            sub_disp = sub_res.headers.get('Content-Disposition', '')
                            if 'filename=' in sub_disp:
                                sub_filename = re.findall(r'filename=["\']?(.*?)["\']?$', sub_disp)[0]
                            else:
                                sub_filename = os.path.basename(urlparse(asset_route).path)
                            
                            # Make sure folder exists and save it inside extracted_documents
                            os.makedirs(doc_folder, exist_ok=True)
                            with open(os.path.join(doc_folder, sub_filename), 'wb') as f_sub:
                                for chunk in sub_res.iter_content(chunk_size=16384): f_sub.write(chunk)
                            print(f"      ✓ Document Saved to sub-folder: {sub_filename}")
                    except Exception: pass

                # Image extraction
                images = soup.find_all('img', src=True)
                if images:
                    os.makedirs(image_folder, exist_ok=True)
                    img_counter = 1
                    MIN_WIDTH, MIN_HEIGHT = 100, 80
                    noise_image_keywords = re.compile(r'(logo|avatar|favicon|loader|spinner|banner-ad|arrow|button|sprite|share-on|social-media)', re.IGNORECASE)
                    
                    for img in images:
                        img_url = urljoin(target_url, img['src'])
                        if img_url.startswith('data:image'): continue
                        
                        parsed_img_path = urlparse(img_url).path
                        img_name_raw = os.path.basename(parsed_img_path)
                        if noise_image_keywords.search(img_name_raw): continue
                            
                        try:
                            with requests.get(img_url, headers=headers, stream=True, timeout=10, verify=False) as img_res:
                                if img_res.ok:
                                    img_data = img_res.content
                                    with Image.open(BytesIO(img_data)) as loaded_img:
                                        width, height = loaded_img.size
                                        if width < MIN_WIDTH or height < MIN_HEIGHT: continue
                                            
                                        img_name = img_name_raw
                                        if not img_name or '.' not in img_name:
                                            ext = f".{loaded_img.format.lower()}" if loaded_img.format else ".jpg"
                                            img_name = f"image_{img_counter}{ext}"
                                            img_counter += 1
                                            
                                        with open(os.path.join(image_folder, img_name), 'wb') as img_f:
                                            img_f.write(img_data)
                                        print(f"      ✓ Image Saved to sub-folder: {img_name} ({width}x{height})")
                        except Exception: pass

    except Exception as e: 
        print(f"   ✗ Connection Interrupted: {e}")

Pipeline

In [7]:
def run_scrapper(url: str) -> str:
    """Pipeline execution wrapper.

    Takes a URL, sanitizes it, runs the dynamic extractor, and returns the
    absolute path where data was written.
    """
    print(f"🚀 Initializing scraper for: {url}")

    # 1. Sanitize input & generate path
    clean_url, target_path = sanitize_and_generate_dynamic_folder(url)

    if not clean_url or not target_path:
        print(" ✗ Error: Invalid URL input provided.")
        return ""

    # 2. Run extraction pipeline
    dynamic_session_extractor(clean_url, target_path)

    print(f" Task finished! Data directory: {os.path.abspath(target_path)}")
    return target_path

In [11]:
%pip install nbformat


   ---------------------------------------- 0/2 [fastjsonschema]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Execution

In [ ]:
%run scraper.ipynb
output_folder = run_scrapper("https:/www.digisinc.systems")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
